# 3D LE network

```bash
python adjacency_3d.py
python comm.py outputs/le3d
```

Figures are saved to `figures/3d/` (3D views as PNG, the rest as SVG).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scienceplots

plt.style.use(["science", "no-latex"])
plt.rcParams["savefig.dpi"] = 600
COLORS = ["#730e22", "#d64e8b", "white", "#068fb9", "#1c3f75"]
CMAP = LinearSegmentedColormap.from_list("modes", COLORS)
import pyvista as pv

pv.global_theme.allow_empty_mesh = True      # a mode may have no isosurface of one sign

CLIM = 0.05        # paper: 0.02 with 18,018 nodes; mode values scale as 1/sqrt(nodes)
# CLIM = 0.02       # for SKIP = 2 in adjacency_3d.py (paper)
OUT = Path("figures/3d")
OUT.mkdir(parents=True, exist_ok=True)

d = np.load("sample/g2_d0.5_101-110.npz")
g = np.load("outputs/le3d/frame_0000110.npz")
c = np.load("outputs/le3d/comm.npz")
x, y, z, foil = d["x"], d["y"], d["z"], d["airfoil"]
bg_z = z[0] - 3.0                            # the z-mean plane is drawn below the wing


def grid(x, y, z):
    return pv.StructuredGrid(*np.meshgrid(x, y, z, indexing="ij"))


def section(zc):
    return pv.PolyData(np.column_stack([foil, np.full(len(foil), zc)]), np.r_[len(foil), np.arange(len(foil))])


def wing(p):
    p.add_mesh(section(z[0]).extrude((0, 0, z[-1] - z[0]), capping=True), color="gray")
    p.add_mesh(section(bg_z + 0.001), color="gray", lighting=False)


def save(p, bars, name):
    """3D view as an image, with colorbars at its lower right."""
    img = p.screenshot(return_img=True)
    p.close()
    drawn = (img < 250).any(axis=2)                          # crop the white margin
    img = img[drawn.any(axis=1)][:, drawn.any(axis=0)]
    fig, ax = plt.subplots()
    ax.imshow(img)
    ax.axis("off")
    for i, (cmap, clim) in enumerate(bars):                   # lower right, next to the image
        cax = ax.inset_axes([1.02 + 0.12 * i, 0, 0.03, 0.2])
        fig.colorbar(plt.cm.ScalarMappable(plt.Normalize(-clim, clim), cmap), cax=cax, ticks=[-clim, clim])
    fig.savefig(OUT / name)


## Flow: Q = 0.1 coloured by $\omega_z$, and $\overline{L_E}^z$ below the wing

In [ ]:
for k in (0, -1):                                # first and last frame of the window
    # lift element on the sample grid (same as lift_element() in adjacency_3d.py)
    u, v, w, omega_z = d["u"][k], d["v"][k], d["w"][k], d["omega_z"][k]
    omega_x = np.gradient(w, y, axis=1) - np.gradient(v, z, axis=2)
    omega_y = np.gradient(u, z, axis=2) - np.gradient(w, x, axis=0)
    le = -((omega_y * w - omega_z * v) * d["grad_phi_x"][:, :, None]
           + (omega_z * u - omega_x * w) * d["grad_phi_y"][:, :, None])

    vol = grid(x, y, z)
    vol["Q"] = d["Q"][k].ravel(order="F")
    vol["omega"] = omega_z.ravel(order="F")
    bg = grid(x, y, [bg_z])
    bg["le"] = le.mean(axis=2).ravel(order="F")
    p = pv.Plotter(off_screen=True, window_size=(2400, 1200))
    p.add_mesh(vol.contour([0.1], scalars="Q"), scalars="omega", cmap="RdBu_r", clim=[-4, 4],
               opacity=0.8, smooth_shading=True, show_scalar_bar=False)
    p.add_mesh(bg, scalars="le", cmap="PiYG_r", clim=[-1, 1], lighting=False, show_scalar_bar=False)
    wing(p)
    p.camera_position = [(-1.7, 2.3, 5.5), (0.75, 0.75, 1), (0, 1, 0)]
    p.camera.zoom(0.9)
    save(p, [("RdBu_r", 4.0), ("PiYG_r", 1.0)], f"flow_frame{d['frames'][k]}.png")


## Forcing (B) and receive (R) modes: $\pm$ isosurfaces and the $z$-mean below the wing

In [ ]:
shape = (len(g["x"]), len(g["y"]), len(g["z"]))
for kind in "BR":
    for m in (1, 2, 3):
        mode = np.zeros(np.prod(shape))
        mode[g["node"]] = c[kind][:, m - 1]
        mode = mode.reshape(shape)
        vol = grid(g["x"], g["y"], g["z"])
        vol["mode"] = mode.ravel(order="F")
        bg = grid(g["x"], g["y"], [bg_z])
        bg["mode"] = mode.mean(axis=2).ravel(order="F")
        p = pv.Plotter(off_screen=True, window_size=(2400, 1200))
        p.add_mesh(vol.contour([CLIM], scalars="mode"), color=COLORS[3], opacity=0.8, smooth_shading=True)
        p.add_mesh(vol.contour([-CLIM], scalars="mode"), color=COLORS[1], opacity=0.8, smooth_shading=True)
        p.add_mesh(bg, scalars="mode", cmap=CMAP, clim=[-CLIM, CLIM], lighting=False, show_scalar_bar=False)
        wing(p)
        p.camera_position = [(-2.7, 2.3, 5.5), (-0.25, 0.75, 1), (0, 1, 0)]
        save(p, [(CMAP, CLIM)], f"{kind}_mode{m}.png")


## Singular values

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(np.arange(1, len(c["S"]) + 1), c["S"], "o-", markersize=3, color="#CC3311")
ax.set(xlabel=r"$k$", ylabel=r"$\sigma_k$")
fig.savefig(OUT / "singular_values.pdf")
